In [2]:
# 1. Instalamos las librerías necesarias
!pip install -q getdist emcee

import numpy as np
import pandas as pd
import scipy.integrate as integrate
import matplotlib.pyplot as plt

print("Cargando los datos...")

# 2. Cargar el "CSV" engañoso con Pandas
nombres_columnas = ['PREFIX', 'CID', 'IDSURVEY', 'zHD', 'zHEL', 'MU', 'MUERR', 'MUERR_VPEC', 'MUERR_SYS', 'PROB1A_BEAMS']

df_des = pd.read_csv('DES-Dovekie_HD.csv',
                     sep=r'\s+',
                     comment='#',
                     names=nombres_columnas
                    )

# Forzamos que las columnas sean números decimales (floats)
df_des['zHD'] = pd.to_numeric(df_des['zHD'], errors='coerce')
df_des['MU'] = pd.to_numeric(df_des['MU'], errors='coerce')

# Borramos las filas vacías
df_des = df_des.dropna(subset=['zHD', 'MU'])

z_obs = df_des['zHD'].values.astype(float)
mu_obs = df_des['MU'].values.astype(float)

# 3. Cargar la Matriz de Covarianza Inversa (.npz)
archivo_npz = np.load('STAT+SYS.npz')

nsn = int(np.squeeze(archivo_npz['nsn']))
inv_cov_cruda = archivo_npz['cov']

# --- EL DESENROLLADOR DEL TRIÁNGULO ---
# Verificamos si el tamaño coincide exactamente con el triángulo superior N*(N+1)/2
if len(inv_cov_cruda.shape) == 1 and len(inv_cov_cruda) == (nsn * (nsn + 1)) // 2:
    print(f"La matriz vino comprimida en triángulo. Reconstruyendo a {nsn}x{nsn}...")

    # 1. Creamos una matriz vacía de NxN
    inv_cov = np.zeros((nsn, nsn))

    # 2. Obtenemos las coordenadas (i, j) del triángulo superior
    i_upper, j_upper = np.triu_indices(nsn)

    # 3. Llenamos el triángulo superior con los datos crudos
    inv_cov[i_upper, j_upper] = inv_cov_cruda

    # 4. Hacemos que sea simétrica copiando lo de arriba hacia abajo
    i_lower, j_lower = np.tril_indices(nsn, -1)
    inv_cov[i_lower, j_lower] = inv_cov.T[i_lower, j_lower]

else:
    # Por si acaso la descargaste en otro formato que ya venía en 2D
    inv_cov = inv_cov_cruda.reshape((nsn, nsn))


print(f"\n¡Éxito! Se cargaron {len(z_obs)} supernovas perfectamente numéricas.")
print(f"Tamaño real de la matriz de covarianza inversa: {inv_cov.shape}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 836.0/836.0 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 3.5 MB/s eta 0:00:00
Cargando los datos...


FileNotFoundError: [Errno 2] No such file or directory: 'STAT+SYS.npz'

In [ ]:
import emcee

# ---------------------------------------------------------
# 1. EL MODELO FÍSICO (Universo Flat Lambda-CDM)
# ---------------------------------------------------------

mB_obs = mu_obs + (-19.36) #****
def E_z(z, Om):
    return np.sqrt(Om * (1 + z)**3 + (1 - Om))

def distancia_luminosidad(z_array, Om, H0):
    c = 299792.458 # km/s #Wikipedia
    ints = np.array([integrate.quad(lambda x: 1.0/E_z(x, Om), 0, zi)[0] for zi in z_array])
    return (c / H0) * (1 + z_array) * ints

# Ahora calculamos la magnitud aparente teórica usando M_abs directamente
def mB_teoria(z_array, Om, H0, M_abs):
    D_L = distancia_luminosidad(z_array, Om, H0)
    return 5 * np.log10(D_L * 1e5) + M_abs

# ---------------------------------------------------------
# 2. LIKELIHOOD
# ---------------------------------------------------------

def log_likelihood(theta, z, mB_obs_data, inv_cov):
    Om, H0, M_abs = theta
    mB_th = mB_teoria(z, Om, H0, M_abs)
    delta = mB_obs_data - mB_th
    chi2 = np.dot(delta.T, np.dot(inv_cov, delta))
    return -0.5 * chi2
# ---------------------------------------------------------
# 3. PRIORS
# ---------------------------------------------------------
def log_prior(theta):
    Om, H0, M_abs = theta

    if not (0.0 < Om < 1.0 and 50.0 < H0 < 100.0 and -21.0 < M_abs < -18.0):
        return -np.inf

    # M = -19.253 +- 0.027
    chi2_SH0ES_M = ((M_abs - (-19.253)) / 0.027)**2 #Riess et al. sección 5.0 pág 33 https://arxiv.org/pdf/2112.04510


    return -0.5 * chi2_SH0ES_M

def log_probability(theta, z, mB_obs_data, inv_cov):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, z, mB_obs_data, inv_cov)

# ---------------------------------------------------------
# 4. CADENA DE MARKOV (MCMC)
# ---------------------------------------------------------
n_walkers = 32 #duplicamos el valor que teníamos antes (16)
n_dim = 3  # [Omega_m, H0, M_abs]
n_steps = 3000 #duplicamos el valor que teníamos antes

# Iniciamos cerca de M = -19.25
pos_inicial = [0.33, 73.0, -19.25] + 1e-4 * np.random.randn(n_walkers, n_dim)

# Pasamos 'mB_obs' en lugar de 'mu_obs'
sampler = emcee.EnsembleSampler(n_walkers, n_dim, log_probability, args=(z_obs, mB_obs, inv_cov))

print(f"Iniciando MCMC de {n_steps} pasos...")
sampler.run_mcmc(pos_inicial, n_steps, progress=True)
print("¡Simulación completada!")

In [ ]:
import getdist
from getdist import MCSamples, plots

# 1. Cortamos el primer 30% como "burn-in"
burn_in = int(0.3 * n_steps)
muestras_planas = sampler.get_chain(discard=burn_in, flat=True)

# 2. Pasamos los datos a GetDist (Columna 0 es Om, Columna 1 es H0)
muestras_para_getdist = np.column_stack((muestras_planas[:, 1], muestras_planas[:, 0]))

chain_des = MCSamples(samples=muestras_para_getdist,
                      names=['H0', 'omegam'],
                      labels=['H_0', r'\Omega_m'])

print("\n--- Resultados Finales (DES + SH0ES) ---")
print(chain_des.getInlineLatex('H0'))
print(chain_des.getInlineLatex('omegam'))

# 3. Graficar el resultado
g = plots.get_subplot_plotter(width_inch=6)
g.triangle_plot([chain_des], ['omegam', 'H0'],
                filled=True, contour_colors=["#D90000"])

plt.suptitle('Cosmología DES-SN5YR calibrada con SH0ES', fontsize=14, y=1.05)
plt.show()

# 4. Cálculo de tu métrica Q_DM frente a Planck (H0 = 67.4 +- 0.5)
H0_des_mean = chain_des.getMeans()[0]
H0_des_var = chain_des.getVars()[0]

Q_DM = ((H0_des_mean - 67.4)**2) / (H0_des_var + 0.5**2)
tension_sigmas = np.sqrt(Q_DM)

print(f"\nLa tensión Q_DM frente a Planck es de: {tension_sigmas:.2f} sigmas")

In [ ]:
# Guardamos la cadena en un archivo .txt separado por espacios
# 'header' añade los nombres de las variables en la primera línea
np.savetxt('cadena_des_shoes_v2.txt', muestras_planas,
           header='Om H0 M_offset', comments='')

print("¡Cadena guardada exitosamente como cadena_des_shoes.txt!")

# Para volver a cargarlo en el futuro:
# muestras_recuperadas = np.loadtxt('cadena_des_shoes.txt', skiprows=1)

In [ ]:
import datetime

print("Preparando los datos y el encabezado...")

# 1. Extraemos el log-posterior (post) directamente del sampler
post_plano = sampler.get_log_prob(discard=burn_in, flat=True)

# 2. Recalculamos el log-prior para cada muestra usando tu función log_prior
# (Como son muchas muestras, esto puede tardar un par de segundos)
prior_plano = np.array([log_prior(theta) for theta in muestras_planas])

# 3. Unimos las 3 columnas de parámetros con las 2 de probabilidades
# El orden será: Om, H0, M_offset, prior, post
datos_completos = np.column_stack((muestras_planas, prior_plano, post_plano))

# 4. Construimos el encabezado personalizado
# Usamos f-strings para inyectar variables reales de tu simulación
fecha_actual = datetime.datetime.now().isoformat()
muestras_totales = len(muestras_planas)

encabezado_cosmosis = f"""cosmological_parameters--omega_m	cosmological_parameters--h0	supernova_params--m	prior	post
#sampler=emcee
#n_varied=3
#timestamp={fecha_actual}
#walkers={n_walkers}
#samples={muestras_totales}
#nsteps={n_steps}
## START_OF_PARAMS_INI
## [runtime]
## sampler = emcee
## likelihood_only = T
##
## [emcee]
## walkers = {n_walkers}
## samples = {muestras_totales}
## nsteps = {n_steps}
##
## [pipeline]
## modules = consistency camb pantheon
## likelihoods = pantheon
## END_OF_PARAMS_INI
## START_OF_VALUES_INI
## [cosmological_parameters]
## omega_m = 0.0  0.33  1.0
## h0 = 50.0  73.0  100.0
##
## [supernova_params]
## m = -5.0  0.0  5.0
## END_OF_VALUES_INI
## START_OF_PRIORS_INI
## [cosmological_parameters]
## h0 = gaussian 73.04 1.04
## END_OF_PRIORS_INI"""

# 5. Guardamos el archivo
# comments='' evita que NumPy ponga un '#' extra al principio del texto
np.savetxt('cadena_des_shoes_cosmosis.txt',
           datos_completos,
           delimiter='\t',
           header=encabezado_cosmosis,
           comments='')

print("¡Listo! Cadena guardada exitosamente como 'cadena_des_shoes_cosmosis.txt'")

#Consideramos Union 3

In [ ]:
!pip install -q astropy getdist emcee

import numpy as np
import emcee
from astropy.io import fits
import datetime
import getdist
from getdist import MCSamples, plots
import matplotlib.pyplot as plt

print("Cargando la matriz de Union3 desde el archivo FITS...")

# Leemos el archivo FITS (asegúrate de que esté subido a Colab)
archivo_fits = 'mu_mat_union3_cosmo=2_mu.fits'
hdul = fits.open(archivo_fits)
datos_fits = hdul[0].data

# La estructura de Rubin:
# Fila 0 (a partir de la columna 1) = redshift (z)
# Columna 0 (a partir de la fila 1) = módulo de distancia (mu)
# El bloque central = Matriz de covarianza inversa
z_obs_u3 = datos_fits[0, 1:]
mu_obs_u3 = datos_fits[1:, 0]
inv_cov_u3 = datos_fits[1:, 1:]

hdul.close()

print(f"¡Éxito! Se cargaron {len(z_obs_u3)} supernovas de Union3.")
print(f"Tamaño de la matriz de covarianza inversa: {inv_cov_u3.shape}")

In [ ]:
# ---------------------------------------------------------
# MCMC PARA UNION3 (Desmarginalizado con Prior de SH0ES)
# ---------------------------------------------------------

# 1. PREPARACIÓN DE DATOS DE UNION3
# Reemplaza 'mu_obs_u3', 'z_obs_u3' e 'inv_cov_u3' por los nombres exactos de tus variables de Union3
# NOTA: Aplica aquí el valor de desempaquetado (M_fiducial) correspondiente a Union3

mB_obs_u3 = mu_obs_u3 + (-19.36)  # O el valor fiducial que corresponda a Union3 tras hacer la prueba

# 2. CONFIGURACIÓN DEL SAMPLER
n_walkers = 32
n_dim = 3      # [Omega_m, H0, M_abs]
n_steps = 3000

# Iniciamos los caminadores cerca del valor físico de SH0ES
pos_inicial_u3 = [0.33, 73.0, -19.25] + 1e-4 * np.random.randn(n_walkers, n_dim)

# 3. EJECUCIÓN (Pasamos las variables de Union3 en 'args')
sampler_u3 = emcee.EnsembleSampler(
    n_walkers,
    n_dim,
    log_probability, # Reutilizamos tu misma función genérica
    args=(z_obs_u3, mB_obs_u3, inv_cov_u3) # ¡Aquí inyectamos los datos de Union3!
)

print(f"Iniciando MCMC para Union3 de {n_steps} pasos...")
sampler_u3.run_mcmc(pos_inicial_u3, n_steps, progress=True)
print("¡Simulación de Union3 completada!")

# 4. EXTRACCIÓN DE CADENAS (Descartando el Burn-in)
# Esta es la variable 'muestras_planas_u3' que luego leerá GetDist
muestras_planas_u3 = sampler_u3.get_chain(discard=800, flat=True)

In [ ]:
print("Generando gráfico de contornos...")

# Extraemos las columnas en el orden correcto para GetDist (H0, omegam)
col_omegam_u3 = muestras_planas_u3[:, 0]
col_H0_u3 = muestras_planas_u3[:, 1]
muestras_getdist_u3 = np.column_stack((col_H0_u3, col_omegam_u3))

# Creamos el objeto de muestras
chain_u3 = MCSamples(samples=muestras_getdist_u3,
                     names=['H0', 'omegam'],
                     labels=['H_0', r'\Omega_m'],
                     name_tag='Union3')

# Resultados rápidos en texto
print("\n--- Resultados Union3 + SH0ES ---")
print(chain_u3.getInlineLatex('H0'))
print(chain_u3.getInlineLatex('omegam'))

# Graficamos el triángulo
g = plots.get_subplot_plotter(width_inch=6)
g.triangle_plot([chain_u3], ['omegam', 'H0'],
                filled=True, contour_colors=["#00A05A"])

plt.suptitle('Cosmología Union3 calibrada con SH0ES', fontsize=14, y=1.05)
plt.show()

In [ ]:
print("Generando gráfico de contornos...")

# 1. Extraemos las 3 columnas en el orden de nuestro MCMC
col_omegam_u3 = muestras_planas_u3[:, 0]
col_H0_u3 = muestras_planas_u3[:, 1]
col_Mabs_u3 = muestras_planas_u3[:, 2] # ¡Nuestra nueva dimensión desmarginalizada!

# Las apilamos en una matriz
muestras_getdist_u3 = np.column_stack((col_omegam_u3, col_H0_u3, col_Mabs_u3))

# 2. Creamos el objeto de muestras agregando M_abs
chain_u3 = MCSamples(samples=muestras_getdist_u3,
                     names=['omegam', 'H0', 'Mabs'],
                     labels=[r'\Omega_m', 'H_0', 'M_{abs}'],
                     name_tag='Union3')

# 3. Resultados rápidos en texto
print("\n--- Resultados Union3 + SH0ES ---")
print("Omega_m: ", chain_u3.getInlineLatex('omegam'))
print("H_0:     ", chain_u3.getInlineLatex('H0'))
print("M_abs:   ", chain_u3.getInlineLatex('Mabs'))

# 4. Graficamos el triángulo completo
g = plots.get_subplot_plotter(width_inch=8) # Más ancho para que respiren las 3 variables
g.triangle_plot([chain_u3], ['omegam', 'H0', 'Mabs'],
                filled=True, contour_colors=["#00A05A"])

plt.suptitle('Cosmología Union3 calibrada con SH0ES (M libre)', fontsize=15, y=1.05)
plt.show()

In [ ]:
print("Guardando las cadenas...")

# ---------------------------------------------------------
# 1. GUARDAR COMO .TXT SIMPLE
# ---------------------------------------------------------
np.savetxt('cadena_union3_shoes.txt', muestras_planas_u3,
           header='Om H0 M_offset', comments='')
print("- Archivo 'cadena_union3_shoes.txt' generado.")

# ---------------------------------------------------------
# 2. GUARDAR COMO .TXT FORMATO COSMOSIS
# ---------------------------------------------------------
# Extraemos el posterior
post_plano_u3 = sampler_u3.get_log_prob(discard=burn_in, flat=True)

# Recalculamos el prior para cada muestra (puede tardar unos segundos)
prior_plano_u3 = np.array([log_prior(theta) for theta in muestras_planas_u3])

# Unimos todo: Om, H0, M_offset, prior, post
datos_cosmosis_u3 = np.column_stack((muestras_planas_u3, prior_plano_u3, post_plano_u3))

fecha_actual = datetime.datetime.now().isoformat()

encabezado_cosmosis_u3 = f"""cosmological_parameters--omega_m	cosmological_parameters--h0	supernova_params--m	prior	post
#sampler=emcee
#n_varied=3
#timestamp={fecha_actual}
#walkers={n_walkers}
#samples={muestras_totales_u3}
#nsteps={n_steps}
## START_OF_PARAMS_INI
## [runtime]
## sampler = emcee
## likelihood_only = T
##
## [emcee]
## walkers = {n_walkers}
## samples = {muestras_totales_u3}
## nsteps = {n_steps}
##
## [pipeline]
## modules = consistency camb pantheon
## likelihoods = pantheon
## END_OF_PARAMS_INI
## START_OF_VALUES_INI
## [cosmological_parameters]
## omega_m = 0.0  0.33  1.0
## h0 = 50.0  73.0  100.0
##
## [supernova_params]
## m = -5.0  0.0  5.0
## END_OF_VALUES_INI
## START_OF_PRIORS_INI
## [cosmological_parameters]
## h0 = gaussian 73.04 1.04
## END_OF_PRIORS_INI"""

np.savetxt('cadena_union3_shoes_cosmosis.txt',
           datos_cosmosis_u3,
           delimiter='\t',
           header=encabezado_cosmosis_u3,
           comments='')
print("- Archivo 'cadena_union3_shoes_cosmosis.txt' generado.")

#Vamos a generar las mismas cadenas de antes pero cambiando el prior de H_0. En lugar de ~  73, a 67 [1/s]

In [ ]:
import emcee
import getdist
from getdist import MCSamples, plots
import numpy as np
import scipy.integrate as integrate
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. EL MODELO FÍSICO Y LIKELIHOOD (DES)
# ---------------------------------------------------------
def E_z(z, Om):
    return np.sqrt(Om * (1 + z)**3 + (1 - Om))

def distancia_luminosidad(z_array, Om, H0):
    c = 299792.458
    ints = np.array([integrate.quad(lambda x: 1.0/E_z(x, Om), 0, zi)[0] for zi in z_array])
    return (c / H0) * (1 + z_array) * ints

def mu_teoria(z_array, Om, H0, M_offset):
    D_L = distancia_luminosidad(z_array, Om, H0)
    return 5 * np.log10(D_L * 1e5) + M_offset

def log_likelihood(theta, z, mu_obs, inv_cov):
    Om, H0, M_offset = theta
    mu_th = mu_teoria(z, Om, H0, M_offset)
    delta_mu = mu_obs - mu_th
    return -0.5 * np.dot(delta_mu.T, np.dot(inv_cov, delta_mu))

# ---------------------------------------------------------
# 2. EL NUEVO PRIOR (¡AQUÍ ENTRA PLANCK!)
# ---------------------------------------------------------
def log_prior_planck(theta):
    Om, H0, M_offset = theta
    if not (0.0 < Om < 1.0 and 50.0 < H0 < 100.0 and -5.0 < M_offset < 5.0):
        return -np.inf

    chi2_Planck = ((H0 - 67.4) / 0.5)**2
    return -0.5 * chi2_Planck

def log_prob_planck(theta, z, mu_obs, inv_cov):
    lp = log_prior_planck(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, z, mu_obs, inv_cov)

# ---------------------------------------------------------
# 3. EJECUTAR EL MCMC (DES + PLANCK)
# ---------------------------------------------------------
n_walkers, n_dim, n_steps = 16, 3, 1500
pos_inicial = [0.33, 67.4, 0.0] + 1e-4 * np.random.randn(n_walkers, n_dim)

print("Iniciando MCMC para DES con prior de PLANCK...")
sampler_des_planck = emcee.EnsembleSampler(n_walkers, n_dim, log_prob_planck, args=(z_obs, mu_obs, inv_cov))
sampler_des_planck.run_mcmc(pos_inicial, n_steps, progress=True)

# ---------------------------------------------------------
# 4. EXTRACCIÓN Y GUARDADO EN ARCHIVO TEXTO (.TXT)
# ---------------------------------------------------------
muestras_des_p = sampler_des_planck.get_chain(discard=int(0.3*n_steps), flat=True)

# Definimos el encabezado para que sepas qué es cada columna al abrir el bloc de notas
encabezado_des = "Omega_m\tH0\tM_offset"

np.savetxt('cadena_des_planck.txt',
           muestras_des_p,
           delimiter='\t',
           header=encabezado_des,
           comments='')

print("- Archivo 'cadena_des_planck.txt' generado y guardado exitosamente.")

# ---------------------------------------------------------
# 5. ANÁLISIS CON GETDIST Y GRÁFICO
# ---------------------------------------------------------
muestras_gd_des_p = np.column_stack((muestras_des_p[:, 1], muestras_des_p[:, 0]))
chain_des_p = MCSamples(samples=muestras_gd_des_p, names=['H0', 'omegam'], labels=['H_0', r'\Omega_m'])

print("\n--- Resultados Finales (DES + PLANCK) ---")
print(chain_des_p.getInlineLatex('H0'))
print(chain_des_p.getInlineLatex('omegam'))

g = plots.get_subplot_plotter(width_inch=6)
g.triangle_plot([chain_des_p], ['omegam', 'H0'], filled=True, contour_colors=["#0055A4"])
plt.suptitle('Cosmología DES-SN5YR calibrada con PLANCK', fontsize=14, y=1.05)
plt.show()

# Cálculo de Tensión inversa (Contra SH0ES: 73.04 +- 1.04)
H0_mean, H0_var = chain_des_p.getMeans()[0], chain_des_p.getVars()[0]
tension_sigmas = np.sqrt(((H0_mean - 73.04)**2) / (H0_var + 1.04**2))
print(f"\nLa tensión frente a SH0ES local es de: {tension_sigmas:.2f} sigmas")

In [ ]:
# ---------------------------------------------------------
# 1. EJECUTAR EL MCMC (UNION3 + PLANCK)
# ---------------------------------------------------------
n_walkers, n_dim, n_steps = 16, 3, 1500
pos_inicial_u3 = [0.33, 67.4, 0.0] + 1e-4 * np.random.randn(n_walkers, n_dim)

print("Iniciando MCMC para Union3 con prior de PLANCK...")
sampler_u3_planck = emcee.EnsembleSampler(n_walkers, n_dim, log_prob_planck, args=(z_obs_u3, mu_obs_u3, inv_cov_u3))
sampler_u3_planck.run_mcmc(pos_inicial_u3, n_steps, progress=True)

# ---------------------------------------------------------
# 2. EXTRACCIÓN Y GUARDADO EN ARCHIVO TEXTO (.TXT)
# ---------------------------------------------------------
muestras_u3_p = sampler_u3_planck.get_chain(discard=int(0.3*n_steps), flat=True)

encabezado_u3 = "Omega_m\tH0\tM_offset"

np.savetxt('cadena_union3_planck.txt',
           muestras_u3_p,
           delimiter='\t',
           header=encabezado_u3,
           comments='')

print("- Archivo 'cadena_union3_planck.txt' generado y guardado exitosamente.")

# ---------------------------------------------------------
# 3. ANÁLISIS CON GETDIST Y GRÁFICO
# ---------------------------------------------------------
muestras_gd_u3_p = np.column_stack((muestras_u3_p[:, 1], muestras_u3_p[:, 0]))
chain_u3_p = MCSamples(samples=muestras_gd_u3_p, names=['H0', 'omegam'], labels=['H_0', r'\Omega_m'])

print("\n--- Resultados Finales (UNION3 + PLANCK) ---")
print(chain_u3_p.getInlineLatex('H0'))
print(chain_u3_p.getInlineLatex('omegam'))

g = plots.get_subplot_plotter(width_inch=6)
g.triangle_plot([chain_u3_p], ['omegam', 'H0'], filled=True, contour_colors=["#008000"])
plt.suptitle('Cosmología Union3 calibrada con PLANCK', fontsize=14, y=1.05)
plt.show()

# Cálculo de Tensión inversa (Contra SH0ES: 73.04 +- 1.04)
H0_mean_u3, H0_var_u3 = chain_u3_p.getMeans()[0], chain_u3_p.getVars()[0]
tension_sigmas_u3 = np.sqrt(((H0_mean_u3 - 73.04)**2) / (H0_var_u3 + 1.04**2))
print(f"\nLa tensión de Union3 frente a SH0ES local es de: {tension_sigmas_u3:.2f} sigmas")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.integrate as integrate

# =========================================================
# 1. CARGAR DATOS DESDE LOS ARCHIVOS
# =========================================================
print("Cargando el catálogo de Supernovas...")

# Nombres de columnas estándar para los catálogos de DES
nombres_columnas = ['PREFIX', 'CID', 'IDSURVEY', 'zHD', 'zHEL', 'MU', 'MUERR', 'MUERR_VPEC', 'MUERR_SYS', 'PROB1A_BEAMS']

# Cargar el archivo de texto/CSV de las supernovas
df_des = pd.read_csv('DES-Dovekie_HD.csv',
                     sep=r'\s+',
                     comment='#',
                     names=nombres_columnas
                    )

# Forzamos que las columnas sean números decimales (floats)
df_des['zHD'] = pd.to_numeric(df_des['zHD'], errors='coerce')
df_des['MU'] = pd.to_numeric(df_des['MU'], errors='coerce')

# Borramos las filas vacías que puedan aparecer por la conversión
df_des = df_des.dropna(subset=['zHD', 'MU'])

z_obs = df_des['zHD'].values
mu_obs = df_des['MU'].values

print("Cargando la matriz de covarianza (STAT+SYS)...")
# Cargar el archivo .npz (probamos mayúsculas y minúsculas por si acaso)
try:
    cov_data = np.load('STAT+SYS.npz')
except FileNotFoundError:
    cov_data = np.load('stat+sys.npz')

# Los archivos .npz son como diccionarios. Extraemos la matriz que viene adentro:
llave = cov_data.files[0]
cov_matrix = cov_data[llave]

# Los errores individuales (Stat+Sys) son la raíz cuadrada de la diagonal de la matriz
mu_err_real = np.sqrt(np.diag(cov_matrix))

print(f"¡Éxito! Se cargaron {len(z_obs)} supernovas con sus errores completos.\n")

# =========================================================
# 2. FUNCIONES TEÓRICAS (MODELO FÍSICO)
# =========================================================
def E_z(z, Om):
    return np.sqrt(Om * (1 + z)**3 + (1 - Om))

def distancia_luminosidad(z_array, Om, H0):
    c = 299792.458 # km/s
    ints = np.array([integrate.quad(lambda x: 1.0/E_z(x, Om), 0, zi)[0] for zi in z_array])
    return (c / H0) * (1 + z_array) * ints

def mu_teoria(z_array, Om, H0, M_offset):
    D_L = distancia_luminosidad(z_array, Om, H0)
    return 5 * np.log10(D_L * 1e5) + M_offset

# =========================================================
# 3. PARÁMETROS DEL MEJOR AJUSTE (Modifica con tus resultados)
# =========================================================
Om_best = 0.329      # Tu Omega_m medido
H0_best = 73.04      # Tu H0 (o el de tu ancla)
M_best = 0.0         # Tu M_offset

# Generamos la línea continua teórica
z_teoria = np.linspace(np.min(z_obs), np.max(z_obs), 200)
mu_modelo = mu_teoria(z_teoria, Om_best, H0_best, M_best)

# =========================================================
# 4. GRAFICAR DIAGRAMA DE HUBBLE Y RESIDUOS
# =========================================================
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True,
                               gridspec_kw={'height_ratios': [3, 1]})

# --- Panel Superior: Datos vs Teoría ---
ax1.errorbar(z_obs, mu_obs, yerr=mu_err_real, fmt='o', color='gray',
             markersize=2.5, ecolor='lightgray', elinewidth=0.8, capsize=0,
             alpha=0.5, label='Datos DES-SN5YR (Stat + Sys)')

ax1.plot(z_teoria, mu_modelo, color='#D90000', linewidth=2.5,
         label=fr'Mejor Ajuste $\Lambda$CDM ($\Omega_m={Om_best:.3f}$, $H_0={H0_best:.1f}$)')

ax1.set_xscale('log')
ax1.set_ylabel(r'Módulo de Distancia ($\mu$)', fontsize=13)
ax1.set_title('Diagrama de Hubble - Supernovas Tipo Ia (DES)', fontsize=15)
ax1.legend(fontsize=12, loc='lower right')
ax1.grid(True, linestyle=':', alpha=0.5)

# --- Panel Inferior: Residuos ---
mu_teoria_puntos = mu_teoria(z_obs, Om_best, H0_best, M_best)
residuos = mu_obs - mu_teoria_puntos

ax2.axhline(0, color='black', linestyle='--', linewidth=1.5)
ax2.errorbar(z_obs, residuos, yerr=mu_err_real, fmt='o', color='gray',
             markersize=2.5, ecolor='lightgray', elinewidth=0.8, alpha=0.5)

ax2.set_xscale('log')
ax2.set_xlabel('Corrimiento al rojo ($z$)', fontsize=13)
ax2.set_ylabel(r'$\Delta \mu$', fontsize=12)
ax2.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.subplots_adjust(hspace=0.05)
plt.show()

In [ ]:
import getdist
from getdist import MCSamples, plots # Use MCSamples directly
import matplotlib.pyplot as plt
import numpy as np
import glob # To find chain files

# Como subiste los archivos sueltos a tu Jupyter, la ruta es directa al prefijo
file_root = "./base_plikHM_TTTEEE_lowl_lowE_lensing"

# Define the expected parameter names and LaTeX labels based on typical Planck chains
# The order here is crucial and assumed to match the columns in the chain files.
# This list should cover all parameters in your chain files, in their correct column order.
# I've included common Planck parameters and those you explicitly access.
param_names = [
    'omegabh2', 'omegach2', 'H0', 'tau', 'ns', 'As', 'theta', 'omegam', 'sigma8'
    # Add other parameters if known from the chain file structure, maintaining order
]
param_labels = [
    r'\Omega_b h^2', r'\Omega_c h^2', r'H_0', r'\tau', r'n_s', r'A_s', r'\theta_{MC}', r'\Omega_m', r'\sigma_8'
]

# --- Manual loading of chain files ---
chain_files = glob.glob(f"{file_root}_*.txt")
chain_data_list = []

if not chain_files:
    raise FileNotFoundError(f"No chain files found for root: {file_root}")

for f in sorted(chain_files): # Ensure consistent order of chains
    print(f"Loading {f}...")
    # Assuming chains are plain text files, space/tab delimited.
    # Getdist chain files typically have: [weight, loglikelihood, param1, param2, ...]
    data = np.loadtxt(f)
    chain_data_list.append(data)

# Concatenate all chain data
all_chain_data = np.vstack(chain_data_list)

# Split into weights, loglikes, and actual parameters
# Assuming the first column is weight, second is loglike, rest are parameters
weights = all_chain_data[:, 0]
loglikes = all_chain_data[:, 1]
parameters = all_chain_data[:, 2:]

# IMPORTANT FIX: Filter 'parameters' to match the length of 'param_names'
# This assumes the provided param_names correspond to the first columns of the chain data.
if parameters.shape[1] != len(param_names):
    print(f"Using only the first {len(param_names)} parameters from the chain data to match provided param_names.")
    parameters = parameters[:, :len(param_names)]

# Apply burn-in by slicing the data *before* creating MCSamples
burn_in_fraction = 0.3
num_total_samples = parameters.shape[0]
burn_in_rows = int(burn_in_fraction * num_total_samples)

print(f"Removed {burn_in_rows} samples ({burn_in_fraction*100:.0f}% burn-in).")

# Create MCSamples directly with the loaded and processed data
samples = MCSamples(
    samples=parameters[burn_in_rows:],
    weights=weights[burn_in_rows:],
    loglikes=loglikes[burn_in_rows:],
    names=param_names,
    labels=param_labels
)

# Extraemos estadísticos detallados
stats = samples.getMargeStats()

print("--- PARÁMETROS COSMOLÓGICOS DERIVADOS (PLANCK 2018) ---")
print(samples.getInlineLatex('H0'))
print(samples.getInlineLatex('omegam'))

# Si quieres ver los 6 parámetros fundamentales que usó la ESA para pasárselos a CAMB:
for param in ['omegabh2', 'omegach2', 'theta', 'tau', 'ns', 'As']:
    # Check if the parameter exists in the loaded samples before trying to access it
    if samples.hasParam(param): # Use hasParam() method to check for parameter existence
        print(f"{param}: {samples.getMeans()[samples.index[param]]:.5f}")
    else:
        print(f"Warning: Parameter '{param}' not found in samples or its name is incorrect.")

In [4]:
!pip install camb
!pip install -q getdist emcee



In [ ]:
import camb
import numpy as np
import matplotlib.pyplot as plt

# 1. Creamos el objeto de parámetros e inyectamos los valores medidos en el PASO 1
pars = camb.CAMBparams()

# Usamos los valores exactos que imprimió GetDist arriba (ejemplo con valores estándar):
pars.set_cosmology(
    cosmomc_theta = 1.04092 / 100.0, # Parámetro 'theta' de GetDist
    ombh2         = 0.02237,         # Parámetro 'omegabh2'
    omch2         = 0.1200,          # Parámetro 'omegach2'
    tau           = 0.0544           # Parámetro 'tau'
)
pars.InitPower.set_params(
    As = 2.1e-9,                     # Parámetro 'As'
    ns = 0.9649                      # Parámetro 'ns'
)
pars.set_for_lmax(2500, lens_potential_accuracy=0)

# 2. Replicamos el cálculo físico (Ecuaciones de Boltzmann)
results = camb.get_results(pars)
powers = results.get_cmb_power_spectra(pars, CMB_unit='muK')
totCL = powers['total']

l = np.arange(totCL.shape[0])
Dl_TT = totCL[:, 0] # Espectro de Temperatura

# 3. Graficamos el espectro de potencia calculado por ti
plt.figure(figsize=(9, 5))
plt.plot(l, Dl_TT, color='#0055A4', lw=2, label=r'Espectro Replicado $\Lambda$CDM (Planck 2018)')
plt.xlabel(r'Multipolo ($l$)', fontsize=12)
plt.ylabel(r'$D_l^{TT}$ [$\mu K^2$]', fontsize=12)
plt.title('Espectro de Potencia de Anisotropías de Temperatura del CMB', fontsize=14)
plt.xlim(2, 2500)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
df_hd = pd.read_csv("/content/DES-Dovekie_HD.csv")
print("Columnas de HD:", df_hd.columns.tolist())

In [ ]:
import pandas as pd
import numpy as np
import emcee
import scipy.integrate as integrate

# =========================================================
# 1. EL MODELO FÍSICO Y LIKELIHOOD
# =========================================================

def E_z(z, Om):
    return np.sqrt(Om * (1 + z)**3 + (1 - Om))

def distancia_luminosidad(z_array, Om, H0):
    c = 299792.458 # km/s (Wikipedia)
    ints = np.array([integrate.quad(lambda x: 1.0/E_z(x, Om), 0, zi)[0] for zi in z_array])
    return (c / H0) * (1 + z_array) * ints

def mB_teoria(z_array, Om, H0, M_abs):
    D_L = distancia_luminosidad(z_array, Om, H0)
    return 5 * np.log10(D_L * 1e5) + M_abs

def log_likelihood(theta, z, mB_obs_data, inv_cov):
    Om, H0, M_abs = theta
    mB_th = mB_teoria(z, Om, H0, M_abs)
    delta = mB_obs_data - mB_th
    chi2 = np.dot(delta.T, np.dot(inv_cov, delta))
    return -0.5 * chi2

def log_prior(theta):
    Om, H0, M_abs = theta
    # Límites físicos lógicos
    if not (0.0 < Om < 1.0 and 50.0 < H0 < 100.0 and -21.0 < M_abs < -18.0):
        return -np.inf

    # Prior de Riess et al. (SH0ES)
    chi2_SH0ES_M = ((M_abs - (-19.253)) / 0.027)**2
    return -0.5 * chi2_SH0ES_M

def log_probability(theta, z, mB_obs_data, inv_cov):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, z, mB_obs_data, inv_cov)

# =========================================================
# 2. CARGAR DATOS Y METADATOS
# =========================================================

# A. Cargar datos cosmológicos (HD)
df_hd = pd.read_csv("/content/DES-Dovekie_HD.csv")
# ¡PASO CRÍTICO! Guardamos el índice original para no perder el orden de la matriz de covarianza
df_hd['indice_original'] = np.arange(len(df_hd))

# B. Cargar Metadatos (Usando tu método exacto para evitar errores de lectura)
ruta_meta = r"/content/DES-Dovekie_Metadata.csv"
columnas = "PREFIX CID CIDint IDSURVEY TYPE FIELD CUTFLAG_SNANA ERRFLAG_FIT zHEL zHELERR zCMB zCMBERR zHD zHDERR VPEC VPECERR LENSDMU LENSDMUERR MWEBV HOST_NMATCH HOST_NMATCH2 HOST_OBJID HOST_ZPHOT HOST_ZPHOTERR HOST_ZSPEC HOST_ZSPECERR HOST_RA HOST_DEC HOST_ANGSEP HOST_DDLR HOST_CONFUSION HOST_LOGMASS HOST_LOGMASS_ERR HOST_LOGSFR HOST_LOGSFR_ERR HOST_LOGsSFR HOST_LOGsSFR_ERR HOST_COLOR HOST_COLOR_ERR HOST_MAG_g HOST_MAG_i HOST_SBFLUXCAL_g HOST_SBFLUXCAL_i HOST_SBMAG_g HOST_SBMAG_i PKMJDINI SNRMAX1 SNRMAX2 SNRMAX3 SNRSUM BANDLIST PKMJD PKMJDERR x1 x1ERR c cERR mB mBERR x0 x0ERR COV_x1_c COV_x1_x0 COV_c_x0 NDOF FITCHI2 FITPROB RADIUS_POP PROB_SCONE PROB_SNIRFV19 PROB_SNNDESCC PROB_SNNJ17 PROB_SNNV19 CUTMASK MU MUMODEL MUERR MUERR_RENORM MUERR_RAW MUERR_VPEC MURES MUPULL M0DIF M0DIFERR CHI2_BEAMS PROBCC_BEAMS biasCor_nevt biasCor_mu biasCorErr_mu biasCor_muCOVSCALE biasCor_muCOVADD IDSAMPLE IZBIN".split()

df_meta = pd.read_csv(ruta_meta,
                      sep=r'\s+',
                      comment='#',
                      names=columnas,
                      skiprows=1,
                      low_memory=False)

# C. Cruzar ambos archivos usando el ID de la supernova ('CID')
df_completo = pd.merge(df_hd, df_meta[['CID', 'HOST_RA', 'HOST_DEC']], on='CID', how='left')

# D. Limpiar coordenadas inválidas y asegurar que sean números (float)
df_completo = df_completo[(df_completo['HOST_RA'] != -999) & (df_completo['HOST_DEC'] != -999)].copy()
df_completo['HOST_RA'] = df_completo['HOST_RA'].astype(float)
df_completo['HOST_DEC'] = df_completo['HOST_DEC'].astype(float)

# =========================================================
# 3. FILTRADO ESPACIAL (Aislar un parche del cielo)
# =========================================================

# Ejemplo: Parche inferior derecho (Ajusta estos valores según tus gráficos)
mascara_parche = (df_completo['HOST_RA'] > 5) & (df_completo['HOST_RA'] < 15) & (df_completo['HOST_DEC'] < -40)

df_parche = df_completo[mascara_parche]
print(f"Total de supernovas en este parche: {len(df_parche)}")

# Extraer vectores de cosmología
z_obs_p = df_parche['zHD'].values
mu_obs_p = df_parche['MU'].values
mB_obs_p = mu_obs_p + (-19.36)

# =========================================================
# 4. RECORTAR LA MATRIZ DE COVARIANZA
# =========================================================

# Cargar la matriz gigante (¡Asegúrate de poner el nombre correcto de tu archivo de covarianza!)
ruta_covarianza = "/content/tu_archivo_de_covarianza.txt" # <-- CAMBIA ESTO
try:
    cov_mat_completa = np.loadtxt(ruta_covarianza)
except FileNotFoundError:
    print(f"ADVERTENCIA: No se encontró la matriz en {ruta_covarianza}. Cambia el nombre en la línea 81.")
    # Matriz dummy (identidad) solo para que el código no colapse si no la cambias
    cov_mat_completa = np.eye(len(df_hd))

# Extraer solo las filas/columnas de las supernovas que sobrevivieron al filtro
indices_a_guardar = df_parche['indice_original'].values
cov_mat_p = cov_mat_completa[np.ix_(indices_a_guardar, indices_a_guardar)]

# Invertir la matriz pequeña
inv_cov_p = np.linalg.inv(cov_mat_p)

# =========================================================
# 5. LANZAR EL MCMC
# =========================================================

n_walkers = 32
n_dim = 3  # [Omega_m, H0, M_abs]
n_steps = 3000

# Iniciamos cerca de M = -19.25, H0 = 73, Om = 0.33
pos_inicial = [0.33, 73.0, -19.25] + 1e-4 * np.random.randn(n_walkers, n_dim)

# Pasamos los datos filtrados al sampler
sampler = emcee.EnsembleSampler(n_walkers, n_dim, log_probability, args=(z_obs_p, mB_obs_p, inv_cov_p))

print(f"Iniciando MCMC de {n_steps} pasos para el parche seleccionado...")
sampler.run_mcmc(pos_inicial, n_steps, progress=True)
print("¡Simulación completada!")

In [ ]:
import pandas as pd
import numpy as np
import emcee
import scipy.integrate as integrate

# =========================================================
# 1. EL MODELO FÍSICO Y LIKELIHOOD
# =========================================================

def E_z(z, Om):
    return np.sqrt(Om * (1 + z)**3 + (1 - Om))

def distancia_luminosidad(z_array, Om, H0):
    c = 299792.458 # km/s (Wikipedia)
    ints = np.array([integrate.quad(lambda x: 1.0/E_z(x, Om), 0, zi)[0] for zi in z_array])
    return (c / H0) * (1 + z_array) * ints

def mB_teoria(z_array, Om, H0, M_abs):
    D_L = distancia_luminosidad(z_array, Om, H0)
    return 5 * np.log10(D_L * 1e5) + M_abs

def log_likelihood(theta, z, mB_obs_data, inv_cov):
    Om, H0, M_abs = theta
    mB_th = mB_teoria(z, Om, H0, M_abs)
    delta = mB_obs_data - mB_th
    chi2 = np.dot(delta.T, np.dot(inv_cov, delta))
    return -0.5 * chi2

def log_prior(theta):
    Om, H0, M_abs = theta
    # Límites físicos lógicos
    if not (0.0 < Om < 1.0 and 50.0 < H0 < 100.0 and -21.0 < M_abs < -18.0):
        return -np.inf

    # Prior de Riess et al. (SH0ES)
    chi2_SH0ES_M = ((M_abs - (-19.253)) / 0.027)**2
    return -0.5 * chi2_SH0ES_M

def log_probability(theta, z, mB_obs_data, inv_cov):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, z, mB_obs_data, inv_cov)

# =========================================================
# 2. CARGAR DATOS Y METADATOS
# =========================================================

# A. Cargar datos cosmológicos (HD)
df_hd = pd.read_csv("/content/DES-Dovekie_HD.csv")
# ¡PASO CRÍTICO! Guardamos el índice original para no perder el orden de la matriz de covarianza
df_hd['indice_original'] = np.arange(len(df_hd))

# B. Cargar Metadatos (Usando tu método exacto para evitar errores de lectura)
ruta_meta = r"/content/DES-Dovekie_Metadata.csv"
columnas = "PREFIX CID CIDint IDSURVEY TYPE FIELD CUTFLAG_SNANA ERRFLAG_FIT zHEL zHELERR zCMB zCMBERR zHD zHDERR VPEC VPECERR LENSDMU LENSDMUERR MWEBV HOST_NMATCH HOST_NMATCH2 HOST_OBJID HOST_ZPHOT HOST_ZPHOTERR HOST_ZSPEC HOST_ZSPECERR HOST_RA HOST_DEC HOST_ANGSEP HOST_DDLR HOST_CONFUSION HOST_LOGMASS HOST_LOGMASS_ERR HOST_LOGSFR HOST_LOGSFR_ERR HOST_LOGsSFR HOST_LOGsSFR_ERR HOST_COLOR HOST_COLOR_ERR HOST_MAG_g HOST_MAG_i HOST_SBFLUXCAL_g HOST_SBFLUXCAL_i HOST_SBMAG_g HOST_SBMAG_i PKMJDINI SNRMAX1 SNRMAX2 SNRMAX3 SNRSUM BANDLIST PKMJD PKMJDERR x1 x1ERR c cERR mB mBERR x0 x0ERR COV_x1_c COV_x1_x0 COV_c_x0 NDOF FITCHI2 FITPROB RADIUS_POP PROB_SCONE PROB_SNIRFV19 PROB_SNNDESCC PROB_SNNJ17 PROB_SNNV19 CUTMASK MU MUMODEL MUERR MUERR_RENORM MUERR_RAW MUERR_VPEC MURES MUPULL M0DIF M0DIFERR CHI2_BEAMS PROBCC_BEAMS biasCor_nevt biasCor_mu biasCorErr_mu biasCor_muCOVSCALE biasCor_muCOVADD IDSAMPLE IZBIN".split()

df_meta = pd.read_csv(ruta_meta,
                      sep=r'\s+',
                      comment='#',
                      names=columnas,
                      skiprows=1,
                      low_memory=False)

# C. Cruzar ambos archivos usando el ID de la supernova ('CID')
df_completo = pd.merge(df_hd, df_meta[['CID', 'HOST_RA', 'HOST_DEC']], on='CID', how='left')

# D. Limpiar coordenadas inválidas y asegurar que sean números (float)
df_completo = df_completo[(df_completo['HOST_RA'] != -999) & (df_completo['HOST_DEC'] != -999)].copy()
df_completo['HOST_RA'] = df_completo['HOST_RA'].astype(float)
df_completo['HOST_DEC'] = df_completo['HOST_DEC'].astype(float)

# =========================================================
# 3. FILTRADO ESPACIAL (Aislar un parche del cielo)
# =========================================================

# Ejemplo: Parche inferior derecho (Ajusta estos valores según tus gráficos)
mascara_parche = (df_completo['HOST_RA'] > 5) & (df_completo['HOST_RA'] < 15) & (df_completo['HOST_DEC'] < -40)

df_parche = df_completo[mascara_parche]
print(f"Total de supernovas en este parche: {len(df_parche)}")

# Extraer vectores de cosmología
z_obs_p = df_parche['zHD'].values
mu_obs_p = df_parche['MU'].values
mB_obs_p = mu_obs_p + (-19.36)

# =========================================================
# 4. RECORTAR LA MATRIZ DE COVARIANZA
# =========================================================

# Cargar la matriz gigante (¡Asegúrate de poner el nombre correcto de tu archivo de covarianza!)
ruta_covarianza = "/content/tu_archivo_de_covarianza.txt" # <-- CAMBIA ESTO
try:
    cov_mat_completa = np.loadtxt(ruta_covarianza)
except FileNotFoundError:
    print(f"ADVERTENCIA: No se encontró la matriz en {ruta_covarianza}. Cambia el nombre en la línea 81.")
    # Matriz dummy (identidad) solo para que el código no colapse si no la cambias
    cov_mat_completa = np.eye(len(df_hd))

# Extraer solo las filas/columnas de las supernovas que sobrevivieron al filtro
indices_a_guardar = df_parche['indice_original'].values
cov_mat_p = cov_mat_completa[np.ix_(indices_a_guardar, indices_a_guardar)]

# Invertir la matriz pequeña
inv_cov_p = np.linalg.inv(cov_mat_p)

# =========================================================
# 5. LANZAR EL MCMC
# =========================================================

n_walkers = 32
n_dim = 3  # [Omega_m, H0, M_abs]
n_steps = 3000

# Iniciamos cerca de M = -19.25, H0 = 73, Om = 0.33
pos_inicial = [0.33, 73.0, -19.25] + 1e-4 * np.random.randn(n_walkers, n_dim)

# Pasamos los datos filtrados al sampler
sampler = emcee.EnsembleSampler(n_walkers, n_dim, log_probability, args=(z_obs_p, mB_obs_p, inv_cov_p))

print(f"Iniciando MCMC de {n_steps} pasos para el parche seleccionado...")
sampler.run_mcmc(pos_inicial, n_steps, progress=True)
print("¡Simulación completada!")

In [5]:
import pandas as pd
import numpy as np
import emcee
import scipy.integrate as integrate
import math

# =========================================================
# 1. EL MODELO FÍSICO Y LIKELIHOOD (Flat Lambda-CDM)
# =========================================================

def E_z(z, Om):
    return np.sqrt(Om * (1 + z)**3 + (1 - Om))

def distancia_luminosidad(z_array, Om, H0):
    c = 299792.458 # km/s
    ints = np.array([integrate.quad(lambda x: 1.0/E_z(x, Om), 0, zi)[0] for zi in z_array])
    return (c / H0) * (1 + z_array) * ints

def mB_teoria(z_array, Om, H0, M_abs):
    D_L = distancia_luminosidad(z_array, Om, H0)
    return 5 * np.log10(D_L * 1e5) + M_abs

def log_likelihood(theta, z, mB_obs_data, inv_cov):
    Om, H0, M_abs = theta
    mB_th = mB_teoria(z, Om, H0, M_abs)
    delta = mB_obs_data - mB_th
    chi2 = np.dot(delta.T, np.dot(inv_cov, delta))
    return -0.5 * chi2

def log_prior(theta):
    Om, H0, M_abs = theta
    if not (0.0 < Om < 1.0 and 50.0 < H0 < 100.0 and -21.0 < M_abs < -18.0):
        return -np.inf
    # Prior gaussiano de SH0ES (Riess et al. 2022)
    chi2_SH0ES_M = ((M_abs - (-19.253)) / 0.027)**2
    return -0.5 * chi2_SH0ES_M

def log_probability(theta, z, mB_obs_data, inv_cov):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, z, mB_obs_data, inv_cov)

# =========================================================
# 2. CARGA AUTOMÁTICA DE DATOS Y METADATOS
# =========================================================

print("Cargando archivos de datos...")

# A. Cargar diagrama de Hubble (HD) auto-detectando las columnas de la línea VARNAMES:
df_hd = pd.read_csv("/content/DES-Dovekie_HD.csv", sep=r'\s+', comment='#')
if 'VARNAMES:' in df_hd.columns:
    df_hd = df_hd.drop(columns=['VARNAMES:'])

# Guardamos el índice original exacto de las filas para no romper la correspondencia con la covarianza
df_hd['indice_original'] = np.arange(len(df_hd))

# B. Cargar Metadatos auto-detectando las columnas de la línea VARNAMES:
df_meta = pd.read_csv("/content/DES-Dovekie_Metadata.csv", sep=r'\s+', comment='#', low_memory=False)
if 'VARNAMES:' in df_meta.columns:
    df_meta = df_meta.drop(columns=['VARNAMES:'])

# C. Limpieza estricta de los IDs ('CID') para asegurar un cruce perfecto
df_hd['CID'] = df_hd['CID'].astype(str).str.strip()
df_meta['CID'] = df_meta['CID'].astype(str).str.strip()

# D. Cruzar ambas tablas para heredar las coordenadas espaciales y el nombre del campo del telescopio
df_completo = pd.merge(df_hd, df_meta[['CID', 'HOST_RA', 'HOST_DEC', 'FIELD']], on='CID', how='left')

# E. Limpieza geométrica de datos nulos o inválidos
df_completo['HOST_RA'] = pd.to_numeric(df_completo['HOST_RA'], errors='coerce')
df_completo['HOST_DEC'] = pd.to_numeric(df_completo['HOST_DEC'], errors='coerce')
df_completo = df_completo[(df_completo['HOST_RA'] != -999) & (df_completo['HOST_DEC'] != -999)]
df_completo = df_completo.dropna(subset=['HOST_RA', 'HOST_DEC'])

print(f"-> Cruce exitoso. {len(df_completo)} supernovas listas con coordenadas válidas.")

# =========================================================
# 3. SELECCIÓN FILTRADA DEL PARCHE (SUB-REGIÓN)
# =========================================================

# MÉTODO A: Filtrado por coordenadas (Ej: Región inferior en Ascensión Recta y Declinación)
mascara_parche = (df_completo['HOST_RA'] > 5) & (df_completo['HOST_RA'] < 15) & (df_completo['HOST_DEC'] < -40)

# MÉTODO B (Opcional): Si prefieres filtrar directamente por un campo específico de DES (ej: los campos E1/E2 de Elais)
# Descomenta la siguiente línea para activar el filtro por campo:
# mascara_parche = df_completo['FIELD'].str.contains('E', na=False)

df_parche = df_completo[mascara_parche]
print(f"==> ¡Supernovas acotadas en tu parche seleccionado: {len(df_parche)} <==")

if len(df_parche) == 0:
    raise ValueError("El parche seleccionado tiene 0 supernovas. Ajusta los rangos del filtro en la Sección 3.")

# Extraer vectores finales de entrada para el MCMC
z_obs_p = df_parche['zHD'].astype(float).values
mu_obs_p = df_parche['MU'].astype(float).values
mB_obs_p = mu_obs_p + (-19.36) # Conversión a magnitud aparente

# =========================================================
# 4. CARGA DINÁMICA Y RECORTE DE LA MATRIZ STAT+SYS (.npz)
# =========================================================

print("Abriendo y analizando estructura de stat+sys.npz...")
# Modifica la ruta aquí si tu archivo tiene otro nombre (ej: 'sts+sys.npz')
d = np.load("/content/STAT+SYS (1).npz")
keys = list(d.keys())

# Extraer la matriz dinámicamente usando llaves comunes
if 'matrix' in keys:
    cov_mat_completa = d['matrix']
elif 'cov' in keys:
    cov_mat_completa = d['cov']
else:
    cov_mat_completa = d[keys[0]]

# DETECTOR INTELIGENTE: Si la matriz viene aplanada en 1D (triángulo superior), la reconstruye a 2D simétrica
if cov_mat_completa.ndim == 1:
    print("-> Detectado formato comprimido (1D). Reconstruyendo matriz simétrica NxN...")
    n_elementos = len(cov_mat_completa)
    n_dimension = int((-1 + math.sqrt(1 + 8 * n_elementos)) / 2)

    matrix_2d = np.zeros((n_dimension, n_dimension))
    matrix_2d[np.triu_indices(n_dimension)] = cov_mat_completa

    # Espejar el triángulo superior al inferior
    i_lower = np.tril_indices(n_dimension, -1)
    matrix_2d[i_lower] = matrix_2d.T[i_lower]
    cov_mat_completa = matrix_2d
else:
    print(f"-> Detectado formato estándar (2D) con dimensiones: {cov_mat_completa.shape}")

# RECORTAR LA COVARIANZA GIGANTE: Extraer solo las filas/columnas de las SNe que sobrevivieron a tu parche
indices_a_guardar = df_parche['indice_original'].values
cov_mat_p = cov_mat_completa[np.ix_(indices_a_guardar, indices_a_guardar)]

# Invertir matemáticamente la sub-matriz adaptada al parche
inv_cov_p = np.linalg.inv(cov_mat_p)
print("-> ¡Matriz de covarianza recortada e invertida con éxito para tu parche!")

# =========================================================
# 5. EJECUCIÓN DEL MCMC (Cadenas de Markov)
# =========================================================

n_walkers = 32
n_dim = 3  # Parámetros: [Omega_m, H0, M_abs]
n_steps = 3000

# Posición inicial compacta alrededor de los valores fiduciales del Universo
pos_inicial = [0.33, 73.0, -19.25] + 1e-4 * np.random.randn(n_walkers, n_dim)

sampler = emcee.EnsembleSampler(n_walkers, n_dim, log_probability, args=(z_obs_p, mB_obs_p, inv_cov_p))

print(f"Iniciando simulación MCMC de {n_steps} pasos...")
sampler.run_mcmc(pos_inicial, n_steps, progress=True)
print("¡Felicidades! Simulación finalizada con éxito. Cadenas listas para graficar esquinas (corner plots) o calcular medias.")

Cargando archivos de datos...
-> Cruce exitoso. 1623 supernovas listas con coordenadas válidas.
==> ¡Supernovas acotadas en tu parche seleccionado: 283 <==
Abriendo y analizando estructura de stat+sys.npz...
-> Detectado formato comprimido (1D). Reconstruyendo matriz simétrica NxN...
-> ¡Matriz de covarianza recortada e invertida con éxito para tu parche!
Iniciando simulación MCMC de 3000 pasos...


100%|██████████| 3000/3000 [18:52<00:00,  2.65it/s]

¡Felicidades! Simulación finalizada con éxito. Cadenas listas para graficar esquinas (corner plots) o calcular medias.


In [7]:
import numpy as np

# =========================================================
# EXPORTAR CADENAS DE MARKOV A TXT
# =========================================================

# 1. Definir el "Burn-in" (Calentamiento)
# Los primeros pasos del MCMC suelen ser "basura" porque los caminantes
# recién están buscando la zona de buena probabilidad. Lo estándar es descartarlos.
pasos_burn_in = 10

# 2. Extraer las cadenas
# flat=True aplasta las dimensiones para que en lugar de tener (pasos, caminantes, variables),
# tengas una sola tabla 2D gigante perfecta para un archivo de texto.
cadenas_planas = sampler.get_chain(discard=pasos_burn_in, flat=True)

# 3. Preparar el archivo de salida
ruta_salida = "/content/cadenas_MCMC_parche.txt"
encabezado = "Omega_m\tH0\tM_abs" # Nombres de las columnas separados por tabulación

# 4. Guardar a disco
np.savetxt(ruta_salida,
           cadenas_planas,
           delimiter='\t',      # Separado por tabulaciones (puedes usar ',' para un csv)
           header=encabezado,
           comments='# ')       # El encabezado empezará con un '#'

print(f"¡Cadenas guardadas con éxito en: {ruta_salida}!")
print(f"Total de muestras (samples) útiles guardadas: {len(cadenas_planas)}")

¡Cadenas guardadas con éxito en: /content/cadenas_MCMC_parche.txt!
Total de muestras (samples) útiles guardadas: 95680


In [8]:
datos_guardados = np.loadtxt("cadenas_MCMC_parche.txt")